# Member 2 — XGBoost (Anomaly Detection on NSL-KDD)

## Architecture Overview

| Stage | Description |
|-------|-------------|
| **Dataset** | NSL-KDD — 41 pre-scaled features (0–1), binary label (0 = normal, 1 = attack) |
| **Preprocessing** | Shared notebook (`00_data_preprocessing.ipynb`) handles loading, EDA, 80/20 stratified split, and **SMOTE** oversampling to balance classes in training data only |
| **Algorithm** | **XGBoost** (Extreme Gradient Boosting) — an ensemble of decision trees trained sequentially, where each tree corrects the errors of the previous ones |
| **Tuning Phase 1** | `RandomizedSearchCV` — explores 50 random combinations across a wide hyperparameter space using 5-fold cross-validation |
| **Tuning Phase 2** | `GridSearchCV` — fine-grained search in a small neighbourhood around the best params found in Phase 1 |
| **Final Training** | Best hyperparameters → train XGBoost with **early stopping** on a held-out validation slice (prevents overfitting) |
| **Evaluation** | Test set metrics: Accuracy, Precision, Recall, F1-Score, ROC-AUC |
| **Explainability** | Feature importance (gain) + SHAP summary plot for global feature attribution |

### How XGBoost Works (Simple)
1. Start with a simple prediction (e.g., average).
2. Build a small decision tree that fixes the biggest mistakes.
3. Add that tree's predictions to the running total.
4. Repeat steps 2–3 for hundreds of rounds, each time correcting residual errors.
5. The final prediction is the **sum** of all trees' outputs.

Key ideas: **gradient boosting** (each tree fits the gradient/error), **regularisation** (`reg_alpha`, `reg_lambda`) to avoid overfitting, and **early stopping** to halt training when validation loss stops improving.

## 1. Import Libraries

In [ ]:
import pickle, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.model_selection import (
    StratifiedKFold, RandomizedSearchCV, GridSearchCV, cross_val_score
)
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, roc_curve
)
from scipy.stats import uniform, randint

try:
    import shap
    SHAP_OK = True
    print('SHAP available.')
except ImportError:
    SHAP_OK = False
    print('SHAP not found. Run: pip install shap')

print('Libraries loaded.')

## 2. Load Preprocessed Data

In [ ]:
with open('../data/processed/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train      = data['X_train']
X_test       = data['X_test']
y_train      = data['y_train']
y_test       = data['y_test']
feature_cols = data['feature_cols']

# After SMOTE, classes are balanced — no need for scale_pos_weight
n_neg = int(np.sum(y_train == 0))
n_pos = int(np.sum(y_train == 1))
print(f'X_train (SMOTE) : {X_train.shape} | normal={n_neg}, attack={n_pos}')
print(f'X_test  (real)  : {X_test.shape}')
print(f'scale_pos_weight: {n_neg/n_pos:.4f} (≈1.0 after SMOTE)')

## 3. Phase 1 — RandomizedSearchCV (Wide)

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_dist = {
    'n_estimators'        : randint(100, 800),
    'max_depth'           : randint(3, 12),
    'learning_rate'       : uniform(0.01, 0.3),
    'subsample'           : uniform(0.6, 0.4),
    'colsample_bytree'    : uniform(0.5, 0.5),
    'colsample_bylevel'   : uniform(0.5, 0.5),
    'min_child_weight'    : randint(1, 10),
    'gamma'               : uniform(0, 0.5),
    'reg_alpha'           : uniform(0, 1.0),
    'reg_lambda'          : uniform(0.5, 2.0),
}

xgb_rand = RandomizedSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=50,
    cv=cv5,
    scoring='f1',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
xgb_rand.fit(X_train, y_train)

print(f'\nBest params (random): {xgb_rand.best_params_}')
print(f'Best CV F1 (random) : {xgb_rand.best_score_:.4f}')

## 4. Phase 2 — GridSearchCV (Fine)

In [ ]:
# Use the best params from Phase 1 as starting point
bp = xgb_rand.best_params_

def rnd(v, step, n=1, low=0.01):  # produce neighbourhood values
    return sorted(set([round(max(low, v + i*step), 4) for i in range(-n, n+1)]))

param_grid = {
    'n_estimators'     : [max(50, bp['n_estimators']-50), bp['n_estimators'], bp['n_estimators']+50],
    'max_depth'        : [max(1, bp['max_depth']-1), bp['max_depth'], bp['max_depth']+1],
    'learning_rate'    : rnd(bp['learning_rate'], 0.02, n=1),
    'subsample'        : [bp['subsample']],
    'colsample_bytree' : [bp['colsample_bytree']],
    'min_child_weight' : [bp['min_child_weight']],
    'reg_alpha'        : rnd(bp['reg_alpha'], 0.1, n=1),
    'reg_lambda'       : rnd(bp['reg_lambda'], 0.2, n=1),
}

xgb_grid = GridSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=cv5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
xgb_grid.fit(X_train, y_train)

best_params = xgb_grid.best_params_
print(f'\nBest params (grid) : {best_params}')
print(f'Best CV F1 (grid)  : {xgb_grid.best_score_:.4f}')

## 5. Train Final XGBoost with Best Params & Early Stopping

In [ ]:
from sklearn.model_selection import train_test_split as _split

# Hold out 15 % of SMOTE-balanced training data for early-stopping validation
X_tr, X_val, y_tr, y_val = _split(X_train, y_train,
                                   test_size=0.15, random_state=42, stratify=y_train)

xgb_final = XGBClassifier(
    **best_params,
    eval_metric='logloss',
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1
)
xgb_final.fit(X_tr, y_tr,
              eval_set=[(X_tr, y_tr), (X_val, y_val)],
              verbose=False)
print(f'Final XGBoost trained — best iteration: {xgb_final.best_iteration}')

## 6. 5-Fold Cross-Validation on Training Set

In [ ]:
cv_f1  = cross_val_score(xgb_final, X_train, y_train, cv=cv5, scoring='f1',       n_jobs=-1)
cv_acc = cross_val_score(xgb_final, X_train, y_train, cv=cv5, scoring='accuracy', n_jobs=-1)
cv_auc = cross_val_score(xgb_final, X_train, y_train, cv=cv5, scoring='roc_auc',  n_jobs=-1)

print('=== 5-Fold Cross-Validation ===')
print(f'F1       : {cv_f1.mean():.4f}  ± {cv_f1.std():.4f}')
print(f'Accuracy : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}')
print(f'ROC-AUC  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, scores, name, color in zip(
    axes, [cv_f1, cv_acc, cv_auc],
    ['F1','Accuracy','ROC-AUC'], ['darkorange','steelblue','crimson']
):
    ax.plot(range(1,6), scores, 'o-', color=color, lw=2, markersize=7)
    ax.axhline(scores.mean(), linestyle='--', color=color, alpha=0.5,
               label=f'Mean={scores.mean():.4f}')
    ax.set_title(f'CV {name}')
    ax.set_xlabel('Fold')
    ax.legend(fontsize=9)
    ax.set_xticks(range(1,6))
plt.suptitle('XGBoost — 5-Fold CV Scores', fontsize=13)
plt.tight_layout()
plt.savefig('cv_scores_xgb.png', dpi=150)
plt.show()

## 7. Test Set Evaluation

In [ ]:
y_pred      = xgb_final.predict(X_test)
y_pred_prob = xgb_final.predict_proba(X_test)[:, 1]

print('=== Test Set Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Normal','Attack']))

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)
roc_auc   = roc_auc_score(y_test, y_pred_prob)

print(f'Accuracy  : {accuracy:.4f}')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'ROC-AUC   : {roc_auc:.4f}')

## 8. Confusion Matrix, ROC Curve & Learning Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Normal','Attack']).plot(
    cmap='Oranges', ax=axes[0])
axes[0].set_title('XGBoost — Confusion Matrix')

fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'XGB (AUC={roc_auc:.4f})')
axes[1].plot([0,1],[0,1],'--', color='navy', lw=1, label='Random')
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].set_title('XGBoost — ROC Curve')
axes[1].legend()
plt.tight_layout()
plt.savefig('cm_roc_xgb.png', dpi=150)
plt.show()

In [ ]:
# Re-use the eval log that was already recorded during final training
log = xgb_final.evals_result()
epochs = len(log['validation_0']['logloss'])

plt.figure(figsize=(8, 4))
plt.plot(range(epochs), log['validation_0']['logloss'], label='Train Loss', color='steelblue')
plt.plot(range(epochs), log['validation_1']['logloss'], label='Val Loss',   color='crimson')
if hasattr(xgb_final, 'best_iteration'):
    plt.axvline(xgb_final.best_iteration, ls='--', color='gray', label=f'Best iter={xgb_final.best_iteration}')
plt.xlabel('Boosting Round')
plt.ylabel('Log Loss')
plt.title('XGBoost — Learning Curve (Train vs Validation)')
plt.legend()
plt.tight_layout()
plt.savefig('learning_curve_xgb.png', dpi=150)
plt.show()

## 9. Feature Importance & SHAP

In [ ]:
feat_df = pd.DataFrame({'Feature': feature_cols,
                        'Importance': xgb_final.feature_importances_})\
            .sort_values('Importance', ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=feat_df.head(15), x='Importance', y='Feature', palette='Oranges_r')
plt.title('XGBoost — Top 15 Feature Importances')
plt.tight_layout()
plt.savefig('feature_importance_xgb.png', dpi=150)
plt.show()

In [ ]:
if SHAP_OK:
    n_shap   = min(1000, len(X_test))
    idx_shap = np.random.RandomState(42).choice(len(X_test), n_shap, replace=False)
    X_shap   = X_test[idx_shap]
    explainer   = shap.TreeExplainer(xgb_final)
    shap_values = explainer.shap_values(X_shap)
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_shap, feature_names=feature_cols, show=False)
    plt.title('XGBoost — SHAP Summary Plot')
    plt.tight_layout()
    plt.savefig('shap_summary_xgb.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('pip install shap to generate SHAP plots.')

## 10. Save Results

In [ ]:
os.makedirs('../comparison', exist_ok=True)
results = pd.DataFrame([{
    'Algorithm':'XGBoost','Type':'Supervised',
    'Accuracy':round(accuracy,4),'Precision':round(precision,4),
    'Recall':round(recall,4),'F1 Score':round(f1,4),'ROC-AUC':round(roc_auc,4),
    'CV F1 Mean':round(cv_f1.mean(),4),'CV F1 Std':round(cv_f1.std(),4)
}])
print(results.to_string(index=False))
results.to_csv('../comparison/results_xgboost.csv', index=False)
print('\nSaved: ../comparison/results_xgboost.csv')